# 08 — Operational Automation & Monitoring

The layer that turns the six modules above into something that survives contact with production: one command that re-runs the whole pipeline, and a drift check that decides when it needs to.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
%matplotlib inline
pd.set_option('display.max_columns', 40)


In [2]:
from monitoring import check_drift

train = pd.read_parquet('../data/processed/train_clean.parquet')
test = pd.read_parquet('../data/processed/test_clean.parquet')
report, any_drift = check_drift(train, test)
report

,feature,ks_statistic,p_value,drift_detected,reference_mean,incoming_mean
0,distance_km,0.005814,0.915720,False,9.926962,9.957845
1,Delivery_person_Ratings,0.008966,0.453389,False,4.636552,4.635775
2,Delivery_person_Age,0.006793,0.791356,False,29.584739,29.538030
3,prep_time_min,0.005614,0.934524,False,9.989801,10.034214
4,traffic_ordinal,0.002893,0.999999,False,1.364815,1.358014


`monitoring.check_drift()` runs a Kolmogorov-Smirnov two-sample test per key feature against the training reference. Any feature with p < 0.01 flips the retraining trigger — this is what a scheduled job would poll before deciding whether to kick off `pipeline.run()`.

In [3]:
print('Retraining trigger:', 'YES — drift detected' if any_drift else 'no drift detected')

Retraining trigger: no drift detected


## End-to-end pipeline

`src/pipeline.py` runs every module above in sequence from raw CSVs to saved artifacts:

```bash
python src/pipeline.py
```

```
[   0.0s] Cleaning raw data...
[   1.9s] Training ETA prediction model (module 1/5)...
[  26.8s]   -> best model: xgboost, MAE=4.18
[  26.8s] Training demand forecasting model (module 2/5)...
[  27.0s]   -> forecast MAE=25.7 orders/day vs naive=183.0
[  27.0s] Building partner profiles for recommendation engine (module 3/5)...
[  27.3s]   -> 1320 partner profiles built
[  27.3s] Scoring fraud/anomaly detection model (module 4/5)...
[  44.2s]   -> flagged 912 anomalous orders (2.0%)
[  44.3s] Running data drift check against holdout test set (module 5/5, monitoring)...
[  44.3s]   -> retraining trigger: no
[  44.3s] Pipeline complete.
```

In a real deployment this would be an Airflow DAG or scheduled CI job, with `check_drift()` as a gate that decides whether the retraining branch runs.